# F1 Race Engineer - AI Dashboard Prototype

This notebook prototypes the backend functionality for the F1 Race Engineer individual assignment. It tests the weather API, local knowledge base search, telemetry loading via FastF1, and tool-calling loop execution before these components are imported into the PyQt6 desktop app.

In [ ]:
# Dependencies installation
# !pip install requests pandas numpy matplotlib openai fastf1

In [ ]:
import json
import requests

# Coords for F1 circuits
TRACK_COORDS = {
    'silverstone': {'lat': 52.0786, 'lon': -1.0169},
    'spa':         {'lat': 50.4372, 'lon':  5.9714},
    'monza':       {'lat': 45.6156, 'lon':  9.2811},
    'nurburgring': {'lat': 50.3356, 'lon':  6.9475},
    'bahrain':     {'lat': 26.0325, 'lon': 50.5106},
    'monaco':      {'lat': 43.7347, 'lon':  7.4206},
    'suzuka':      {'lat': 34.8431, 'lon': 136.5407},
    'interlagos':  {'lat': -23.7036, 'lon': -46.6997},
    'australia':   {'lat': -37.8497, 'lon': 144.9680},
    'barcelona':   {'lat': 41.5700, 'lon':  2.2611},
}

def get_track_weather(track_name):
    # Fetch weather at circuit from Open-Meteo API
    key = track_name.lower().replace(' ', '').replace('-', '')
    
    matched = None
    for k in TRACK_COORDS:
        if k in key or key in k:
            matched = k
            break
            
    if not matched:
        return json.dumps({"error": f"Unknown track '{track_name}'"})
        
    lat = TRACK_COORDS[matched]['lat']
    lon = TRACK_COORDS[matched]['lon']
    url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current=temperature_2m,rain,surface_temperature,wind_speed_10m,relative_humidity_2m"
    
    try:
        r = requests.get(url, timeout=10)
        data = r.json()['current']
        is_wet = 'WET' if data['rain'] > 0 else 'DRY'
        return json.dumps({
            'circuit': matched,
            'air_temp_c': data['temperature_2m'],
            'track_surface_temp_c': data['surface_temperature'],
            'rain_mm': data['rain'],
            'wind_speed_kmh': data['wind_speed_10m'],
            'humidity_pct': data['relative_humidity_2m'],
            'conditions': is_wet
        })
    except Exception as e:
        return json.dumps({"error": str(e)})

print(get_track_weather('spa'))

In [ ]:
import re

def search_knowledge_base(query, kb_content):
    # Split the knowledge base into markdown sections
    sections = re.split(r'\n(?=## )', kb_content)
    query_words = set(re.sub(r'[^\w\s]', '', query.lower()).split())
    
    scored = []
    for s in sections:
        score = 0
        for w in query_words:
            if len(w) > 2 and w in s.lower():
                score += 1
        scored.append((score, s.strip()))
        
    # Sort best matches first
    scored.sort(key=lambda x: x[0], reverse=True)
    top = [s for score, s in scored if score > 0][:2]
    
    if top:
        return '\n\n---\n\n'.join(top)
    return "No matches found."

test_kb = """
## Pirelli Tyre Compounds
Soft C5 is for qualifying, Medium C3 is for race stints, Hard C2 is for undercut defense.
"""
print(search_knowledge_base('what is the Soft compound for?', test_kb))

In [ ]:
import fastf1
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# Test FastF1 telemetry download and plotting
os.makedirs('f1_cache', exist_ok=True)
fastf1.Cache.enable_cache('f1_cache')

try:
    session = fastf1.get_session(2024, 'Belgian Grand Prix', 'R')
    session.load(telemetry=True, laps=True)
    fastest = session.laps.pick_fastest()
    
    car_data = fastest.get_car_data().add_distance()
    df = pd.DataFrame({
        'Distance_m': car_data['Distance'],
        'Speed_kmh': car_data['Speed'],
        'Throttle_pct': car_data['Throttle'],
        'Brake_pct': car_data['Brake'].astype(int) * 100
    })
    
    plt.figure(figsize=(10, 4))
    plt.plot(df['Distance_m'], df['Speed_kmh'], label='Speed (km/h)', color='cyan')
    plt.xlabel('Distance (m)')
    plt.ylabel('Speed')
    plt.title(f"Fastest lap telemetry test - {fastest['Driver']}")
    plt.legend()
    plt.grid(True)
    plt.show()
except Exception as e:
    print(f"Could not load telemetry: {e}. Plotting dummy chart instead.")
    dist = np.linspace(0, 5000, 200)
    speed = 250 + 50 * np.sin(dist / 400)
    plt.plot(dist, speed, color='orange')
    plt.title('Demo Chart (Offline Fallback)')
    plt.show()

In [ ]:
from openai import OpenAI

# Tool definitions for function calling
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_track_weather",
            "description": "Gets weather details at a circuit",
            "parameters": {
                "type": "object",
                "properties": {
                    "track_name": {"type": "string"}
                },
                "required": ["track_name"]
            }
        }
    }
]

def test_assistant_call(prompt, api_key=None):
    # Simple function to check connectivity and fallback paths
    use_openai = False
    try:
        r = requests.get('http://localhost:11434/api/tags', timeout=2)
        if r.status_code == 200:
            client = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
            model = 'llama3.1:8b'
            print("Connected to local Ollama server.")
        else:
            use_openai = True
    except Exception:
        use_openai = True
        
    if use_openai:
        if not api_key:
            print("Ollama not running and no OpenAI API key provided. Skipping conversation test.")
            return
        client = OpenAI(api_key=api_key)
        model = 'gpt-4o'
        print("Connected to OpenAI.")
        
    res = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        tools=TOOLS,
        tool_choice="auto"
    )
    print("AI Response:", res.choices[0].message.content or "(Empty response - likely tool call request)")
    if res.choices[0].message.tool_calls:
        print("AI wants to run tool calls:", res.choices[0].message.tool_calls)

### Summary

All prototypes are clean, simple, and functional. The logic matches the PyQt6 desktop app modules.